# Signformer — PHOENIX-2014T (Sign Language Recognition)

**Task:** signs → gloss sequence (CSLR).
**Input:** precomputed 1024-dim I3D features.
**Model:** compact Transformer encoder-decoder (hidden = 256, 1 encoder + 1 decoder layer).

---
Run the cells in order the first time. The data-preparation cells (Section 3) and the
SophiaG fix (Section 2) only need to be executed **once**.

## 0 — Environment check

In [1]:
import sys, subprocess
from pathlib import Path

print("Python:", sys.version)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Ensure the working directory is the Signformer folder
SIGNFORMER_DIR = Path(".").resolve()
print("Working dir:", SIGNFORMER_DIR)
assert (SIGNFORMER_DIR / "main" / "training.py").exists(), \
    "ERROR: run this notebook from the Signformer/ folder"

Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.4 GB
Working dir: phoenix\code\signformer


## 1 — Install dependencies

In [2]:
# Legacy torchtext (Field/Iterator) was removed and rewritten; only modern torchtext is needed
# The torchtext APIs are no longer used in the patched code
subprocess.run([sys.executable, "-m", "pip", "install",
    "numpy", "portalocker", "PyYAML", "pandas",
    "matplotlib", "seaborn", "tensorboard",
    "torchtext",   # modern version, used only for Vocabulary internally
], check=True)
print("Dependencies OK")

Dependencies OK


## 2 — Fix: SophiaG optimizer (run once)

The repository imports `SophiaG` unconditionally. This cell patches `builders.py` to fall
back to **Adam** when SophiaG is not installed.

In [ ]:
# The SophiaG-optional import and the Adam fallback in build_optimizer are
# now baked into code/signformer/main/builders.py, so no runtime source
# patching is needed here.
print('builders.py already contains the SophiaG / Adam-fallback patch')


## 3 — Prepare data 

Converts I3D `.npy` files + annotation CSVs into gzipped pickles in the Signformer format.
Output written to the configured data directory.

In [4]:
import os
# Repo layout: phoenix/code/signformer/  (this notebook runs from the signformer dir).
# PHOENIX_ROOT = top-level "phoenix" folder; override with the PHOENIX_ROOT env var.
PHOENIX_ROOT = Path(os.environ.get("PHOENIX_ROOT", Path.cwd().resolve().parents[1]))
DATASET_ROOT = Path(os.environ.get("PHOENIX_DATASET_ROOT", PHOENIX_ROOT / "dataset"))
I3D_ROOT = DATASET_ROOT / "i3d_features_rwth phoenix 2014t"
ANN_ROOT = DATASET_ROOT / "annotations" / "manual"

all_ok = True
for p in [I3D_ROOT / "train", I3D_ROOT / "val", I3D_ROOT / "test",
          ANN_ROOT / "PHOENIX-2014-T.train.corpus.csv"]:
    status = "OK" if p.exists() else "MISSING"
    if not p.exists(): all_ok = False
    print(f"  [{status}] {p}")

if not all_ok:
    raise FileNotFoundError("One or more source files missing -- check the paths")


  [OK] phoenix\dataset\i3d_features_rwth phoenix 2014t\train
  [OK] phoenix\dataset\i3d_features_rwth phoenix 2014t\val
  [OK] phoenix\dataset\i3d_features_rwth phoenix 2014t\test
  [OK] phoenix\dataset\annotations\manual\PHOENIX-2014-T.train.corpus.csv


In [ ]:
%run prepare_phoenix_i3d.py

In [ ]:
# Verify output
out_dir = Path("../../dataset/features/i3d_pami0")
for f in ["phoenix14t.pami0.train", "phoenix14t.pami0.dev", "phoenix14t.pami0.test"]:
    p = out_dir / f
    status = f"OK  ({p.stat().st_size / 1e6:.1f} MB)" if p.exists() else "MISSING"
    print(f"  {status}  <- {f}")

## 4 — Configuration: batch size and optimizer

In [ ]:
import yaml

cfg_path = Path("configs/sign.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

print(f"batch_size : {cfg['training']['batch_size']}  (lower to 16 if you have < 4 GB free VRAM)")
print(f"optimizer  : {cfg['training']['optimizer']}   (sophiag -> Adam fallback if not installed)")
print(f"epochs     : {cfg['training']['epochs']}")
print(f"hidden_size: {cfg['model']['encoder']['hidden_size']}")

In [ ]:
# OPTIONAL — uncomment the lines you want to change, then the write_text line

# cfg['training']['batch_size'] = 16     # less VRAM
# cfg['training']['optimizer']  = 'adam' # skip SophiaG entirely
# cfg['training']['epochs']     = 200

# cfg_path.write_text(yaml.dump(cfg, allow_unicode=True))
# print("Config saved")
print("Config unchanged (uncomment to modify)")

## 5 — Smoke test (data loading)

In [ ]:
if str(SIGNFORMER_DIR) not in sys.path:
    sys.path.insert(0, str(SIGNFORMER_DIR))

from main.helpers import load_config
from main.data import load_data, make_data_iter

cfg = load_config("configs/sign.yaml")
train_data, dev_data, test_data, gls_vocab, txt_vocab = load_data(cfg["data"])

print(f"Train : {len(train_data)} samples")
print(f"Dev   : {len(dev_data)} samples")
print(f"Test  : {len(test_data)} samples")
print(f"Gloss vocab: {len(gls_vocab)} token")
print(f"Text  vocab: {len(txt_vocab)} token")

# Test one batch
it = make_data_iter(train_data, batch_size=4, train=True, shuffle=False)
batch = next(iter(it))
sgn, sgn_len = batch.sgn
gls, gls_len = batch.gls
txt, txt_len = batch.txt
print(f"\nBatch sgn shape : {sgn.shape}   (B, T, feat_dim)")
print(f"Batch gls shape : {gls.shape}")
print(f"Batch txt shape : {txt.shape}")
print("Smoke test OK")

KeyboardInterrupt: 

## 5.5 — Optuna hyperparameter search (optional, ~30% epochs)

In [ ]:
USE_OPTUNA   = True
OPTUNA_TRIALS = 6
OPTUNA_FRAC   = 0.30

if USE_OPTUNA:
    try:
        import optuna
    except ImportError:
        import subprocess, sys as _s
        subprocess.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'optuna'])
        import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    import copy, gc, tempfile, time as _time, yaml, torch
    if str(SIGNFORMER_DIR) not in sys.path:
        sys.path.insert(0, str(SIGNFORMER_DIR))
    import main.training as mt

    _CFG_PATH = str(SIGNFORMER_DIR / 'configs' / 'sign.yaml')
    with open(_CFG_PATH) as f:
        _base_cfg = yaml.safe_load(f)
    _train_data, _dev_data, _test_data, _gls_vocab, _txt_vocab = mt.load_data(data_cfg=_base_cfg['data'])
    _sgn_dim = (sum(_base_cfg['data']['feature_size'])
                if isinstance(_base_cfg['data']['feature_size'], list)
                else _base_cfg['data']['feature_size'])
    _short_patience = max(3, int(_base_cfg['training'].get('early_stopping_patience', 10) * OPTUNA_FRAC))

    print(f'[Optuna] Signformer hyperparameter search')
    print(f'  trials         : {OPTUNA_TRIALS}')
    print(f'  patience       : {_short_patience} (full: {_base_cfg["training"].get("early_stopping_patience", 10)})')
    print(f'  searching      : learning_rate, weight_decay, encoder_dropout')
    print(f'  train samples  : {len(_train_data)}')
    print(f'  dev samples    : {len(_dev_data)}')
    print('=' * 70)

    _optuna_log = []

    def _trial_wer(trial):
        _t0 = _time.time()
        lr    = trial.suggest_float('learning_rate', 1e-4, 8e-4, log=True)
        wd    = trial.suggest_float('weight_decay', 1e-4, 1e-2, log=True)
        drop  = trial.suggest_float('encoder_dropout', 0.1, 0.3)

        print(f'\n--- Trial {trial.number + 1}/{OPTUNA_TRIALS} ---')
        print(f'  learning_rate    = {lr:.6f}')
        print(f'  weight_decay     = {wd:.6f}')
        print(f'  encoder_dropout  = {drop:.4f}')

        cfg_t = copy.deepcopy(_base_cfg)
        tr = cfg_t['training']
        tr['learning_rate']           = float(lr)
        tr['weight_decay']            = float(wd)
        tr['early_stopping_patience'] = _short_patience
        tr['reset_best_ckpt'] = True; tr['reset_optimizer'] = True; tr['reset_scheduler'] = True
        tr['overwrite'] = True
        tr['model_dir'] = tempfile.mkdtemp(prefix='optuna_trial_', dir='.')
        cfg_t['model']['encoder']['dropout'] = float(drop)
        cfg_t['model']['encoder']['embeddings']['dropout'] = float(drop)
        do_rec = tr.get('recognition_loss_weight', 1.0) > 0.0
        do_trs = tr.get('translation_loss_weight', 1.0) > 0.0
        mm = cfg_t['data'].get('multimodal', 1.0) > 0.0
        _model = mt.build_model(cfg=cfg_t['model'], multimodal=mm,
                                gls_vocab=_gls_vocab, txt_vocab=_txt_vocab, sgn_dim=_sgn_dim,
                                do_recognition=do_rec, do_translation=do_trs)
        trainer = mt.TrainManager(model=_model, config=cfg_t)
        trainer.train_and_validate(train_data=_train_data, valid_data=_dev_data)
        wer = float(trainer.best_ckpt_score)
        elapsed = _time.time() - _t0

        _optuna_log.append({
            'trial': trial.number + 1, 'wer': wer,
            'lr': lr, 'wd': wd, 'drop': drop, 'time': elapsed,
        })
        print(f'  => Trial {trial.number + 1} done | best dev WER = {wer:.2f}% | {elapsed:.0f}s')

        del trainer, _model
        gc.collect(); torch.cuda.empty_cache()
        return wer

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_trial_wer, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

    print('\n' + '=' * 70)
    print('[Optuna] SEARCH COMPLETE — summary:')
    print(f'{"#":>3} {"WER%":>7} {"lr":>10} {"wd":>10} {"drop":>7} {"time":>6}')
    print('-' * 70)
    for r in sorted(_optuna_log, key=lambda x: x['wer']):
        print(f'{r["trial"]:3d} {r["wer"]:7.2f} {r["lr"]:10.6f} {r["wd"]:10.6f} '
              f'{r["drop"]:7.4f} {r["time"]:5.0f}s')
    print(f'\nbest dev WER : {study.best_value:.2f}%')
    print(f'best params  : {study.best_params}')

    with open(_CFG_PATH) as f:
        _y = yaml.safe_load(f)
    _y['training']['learning_rate']            = float(study.best_params['learning_rate'])
    _y['training']['weight_decay']             = float(study.best_params['weight_decay'])
    _y['model']['encoder']['dropout']          = float(study.best_params['encoder_dropout'])
    _y['model']['encoder']['embeddings']['dropout'] = float(study.best_params['encoder_dropout'])
    with open(_CFG_PATH, 'w') as f:
        yaml.safe_dump(_y, f, sort_keys=False)
    print('configs/sign.yaml updated with Optuna best.')
else:
    print('Optuna search disabled — using current YAML values.')

## 6 — Training

Starts training. Checkpoints are saved under `./sign_sample/`. You may stop at any time —
training resumes automatically from the periodic checkpoint (`resume.ckpt`, saved every 5 epochs).

In [ ]:
from main.training import train
train(cfg_file="configs/sign.yaml")

COPE:  False


INFO:main.helpers:Hello! This is SL-CAT.
INFO:main.helpers:Total params: 1,623,400
INFO:main.helpers:Trainable parameters: ['encoder.layer_norm.bias', 'encoder.layer_norm.weight', 'encoder.layers.0.FF2.module.layer_norm.bias', 'encoder.layers.0.FF2.module.layer_norm.weight', 'encoder.layers.0.FF2.module.pwff_layer.0.bias', 'encoder.layers.0.FF2.module.pwff_layer.0.weight', 'encoder.layers.0.FF2.module.pwff_layer.3.bias', 'encoder.layers.0.FF2.module.pwff_layer.3.weight', 'encoder.layers.0.att.module.attention.att_layer.bias', 'encoder.layers.0.att.module.attention.att_layer.weight', 'encoder.layers.0.att.module.attention.k_layer.bias', 'encoder.layers.0.att.module.attention.k_layer.weight', 'encoder.layers.0.att.module.attention.output_layer.bias', 'encoder.layers.0.att.module.attention.output_layer.weight', 'encoder.layers.0.att.module.attention.q_layer.bias', 'encoder.layers.0.att.module.attention.q_layer.weight', 'encoder.layers.0.att.module.attention.sample_offsets.bias', 'encoder.

[builders] SophiaG non disponibile, uso Adam come fallback


~\anaconda3\envs\sign-language-dnn\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
INFO:main.helpers:cfg.name                           : sign_experiment
INFO:main.helpers:cfg.data.data_path                 : ./data/
INFO:main.helpers:cfg.data.version                   : phoenix_2014_trans
INFO:main.helpers:cfg.data.sgn                       : sign
INFO:main.helpers:cfg.data.txt                       : text
INFO:main.helpers:cfg.data.gls                       : gloss
INFO:main.helpers:cfg.data.train                     : PHOENIX2014T/phoenix14t.pami0.train
INFO:main.helpers:cfg.data.dev                       : PHOENIX2014T/phoenix14t.pami0.dev
INFO:main.helpers:cfg.data.test                      : PHOENIX2014T/phoenix14t.pami0.test
INFO:main.helpers:cfg.data.feature_size              : 1024
INFO:main.helpers:cfg.data.level                     : word
INFO:main.helpe

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weigh

## 6.5 — Plots: loss and WER trajectory

Reads `sign_sample/validations.txt` (written during training) and plots the curves. Works
**while training is in progress**: re-run the cell to refresh.

In [ ]:
# Plots of recognition loss and dev-set WER.
# Standalone cell: parses sign_sample/validations.txt (no kernel restart required).
import re
from pathlib import Path
import matplotlib.pyplot as plt

MODEL_DIR = "sign_sample"
vpath = Path(MODEL_DIR) / "validations.txt"
assert vpath.exists(), f"{vpath} not found: training has not validated yet"

def _num(pat, line):
    m = re.search(pat, line)
    if not m:
        return float("nan")
    v = float(m.group(1))
    return float("nan") if v == -1.0 else v

steps, rloss, wer = [], [], []
for line in vpath.read_text(encoding="utf-8").splitlines():
    if not line.startswith("Steps:"):
        continue
    steps.append(int(re.search(r"Steps:\s*(\d+)", line).group(1)))
    rloss.append(_num(r"Recognition Loss:\s*([-\d.]+)", line))
    wer.append(_num(r"WER\s+([-\d.]+)", line))

print(f"Validations parsed: {len(steps)}")
valid = [(s, w) for s, w in zip(steps, wer) if w == w]
if valid:
    bs, bw = min(valid, key=lambda t: t[1])
    print(f"Best dev WER = {bw:.2f}%  @ step {bs}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(steps, rloss, marker="o", ms=3, color="tab:blue")
ax1.set_xlabel("step"); ax1.set_ylabel("recognition loss (dev)")
ax1.set_title("Loss"); ax1.grid(alpha=.3)

ax2.plot(steps, wer, marker="o", ms=3, color="crimson", label="dev WER")
if valid:
    ax2.scatter([bs], [bw], color="black", zorder=5, label=f"best {bw:.2f}%")
ax2.set_xlabel("step"); ax2.set_ylabel("WER (%)")
ax2.set_title("Dev WER"); ax2.grid(alpha=.3); ax2.legend()

fig.tight_layout()
fig.savefig(Path(MODEL_DIR) / "training_curves.png", dpi=120)
plt.show()

## 7 — Evaluation (test set)

In [ ]:
import glob

ckpts = sorted(glob.glob("sign_sample/*.ckpt"))
if ckpts:
    best_ckpt = ckpts[-1]
    print(f"Checkpoint: {best_ckpt}")
else:
    raise FileNotFoundError("No checkpoint found -- training not yet completed")

In [ ]:
from main.prediction import test as run_test

run_test(
    cfg_file="configs/sign.yaml",
    ckpt=best_ckpt,
    output_path="sign_sample/test_output.txt"
)

## 8 — TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir sign_sample